In [1]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

/home/shanks/Projects/ai-taste/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Download and example reference data point from LangSmith

In [2]:
ls_client = Client()

In [3]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [4]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('eafc3041-5105-4a4d-b78b-afb498b3931d'), created_at=datetime.datetime(2026, 4, 2, 17, 12, 25, 808616, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 4, 2, 17, 12, 25, 808616, tzinfo=TzInfo(0)), example_count=32, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'sdk_version': '0.7.24', 'runtime_version': '3.12.3', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [7]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[0].outputs

{'ground_truth': 'For cleaning AirPods, the 4-in-1 AirPods Cleaner Pen (B0B7495RL6) is suitable; for organizing cables, the reusable silicone cable ties 40-pack (B0B55TZLBR) would help keep charging cords tidy.',
 'reference_context_ids': ['B0B7495RL6', 'B0B55TZLBR'],
 'reference_descriptions': ['Cleaner Kit for AirPods Pro, 4 in 1 Earbuds Cleaning Pen, Bluetooth Headphone Cleaning Pen for Airpods, Airpods Pro 1 2 3 and Other Earphones, Keyboard, Mouse, Cellphones, Laptop, Camera (White) 【4-IN-1 DESIGN】The airpod cleaning kit is divided into 4 parts - flocking sponge, high-density brush, long-bristle brush and metal tip, which can deeply clean the earbuds and earphone charging compartment at different angles, making your earphones like new 【MULTIFUNCTIONAL CLEANING KIT】The metal tip can precisely remove the dirt in the sound hole of the earbuds, and the high-density brush under the cleaning pen tip is used to clean the fine dust. There are also flocking sponge ends and long-bristle bru

In [8]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[0].inputs

{'question': 'Can you recommend products for cleaning AirPods and organizing their cables?'}

In [9]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[0].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[0].outputs

### RAG Pipeline

In [12]:

def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [13]:
rag_pipeline("Can I get some charger?", top_k=5)

{'answer': 'Yes. We have a few charger cable options available:\n\n1) iPhone Lightning cables (Apple MFi Certified)\n- iPhone Charger Cord Lightning Cables, Original 2022 Upgraded [3Pack 3ft] (ID: B0BYYLJRHT) – Black\n- MUXA 6 Pack [Apple MFi Certified] iPhone Lightning cables (multiple lengths: 3/3/6/6/10/10 FT) (ID: B09TNXY54Y)\n- GREPHONE 2 Pack USB C to Lightning Cable, 6 ft (ID: B0BV6PWVCG)\n\n2) Universal multi-cable (for different connector types)\n- 5 in 1 USB C to Multi Charging Cable, 10 ft (ID: B0BFPZGYLD) – supports Lightning + Type C + Micro USB (note: says not for iPad/tablets)\n\nTell me what device you’re charging (iPhone model or connector type) and how long you want the cable, and I’ll point you to the best match.',
 'question': 'Can I get some charger?',
 'retrieved_context_ids': ['B0BYYLJRHT',
  'B0BFPZGYLD',
  'B09TNXY54Y',
  'B0BV6PWVCG',
  'B0BGDQLZD2'],
 'retrieved_context': ['iPhone Charger Cord Lightning Cables, Original 2022 Upgraded [3Pack 3ft] Apple MFi Cer

### RAGAS Metrics

In [14]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/tmp/ipykernel_7343/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/tmp/ipykernel_7343/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/tmp/ipykernel_7343/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faith

In [15]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/tmp/ipykernel_7343/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/tmp/ipykernel_7343/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [16]:
reference_input

{'question': 'Can you recommend products for cleaning AirPods and organizing their cables?'}

In [17]:
reference_output

{'ground_truth': 'For cleaning AirPods, the 4-in-1 AirPods Cleaner Pen (B0B7495RL6) is suitable; for organizing cables, the reusable silicone cable ties 40-pack (B0B55TZLBR) would help keep charging cords tidy.',
 'reference_context_ids': ['B0B7495RL6', 'B0B55TZLBR'],
 'reference_descriptions': ['Cleaner Kit for AirPods Pro, 4 in 1 Earbuds Cleaning Pen, Bluetooth Headphone Cleaning Pen for Airpods, Airpods Pro 1 2 3 and Other Earphones, Keyboard, Mouse, Cellphones, Laptop, Camera (White) 【4-IN-1 DESIGN】The airpod cleaning kit is divided into 4 parts - flocking sponge, high-density brush, long-bristle brush and metal tip, which can deeply clean the earbuds and earphone charging compartment at different angles, making your earphones like new 【MULTIFUNCTIONAL CLEANING KIT】The metal tip can precisely remove the dirt in the sound hole of the earbuds, and the high-density brush under the cleaning pen tip is used to clean the fine dust. There are also flocking sponge ends and long-bristle bru

In [18]:
result = rag_pipeline(reference_input["question"])

In [19]:
result

{'answer': 'For cleaning AirPods, the Cleaner Kit for AirPods Pro (4-in-1 Earbuds Cleaning Pen) (ID: B0B7495RL6, rating 4.3) is designed for earbuds and the earphone charging compartment. It includes a metal tip to remove dirt in the sound hole, plus different brush/sponge ends for fine dust and angled cleaning.\n\nFor organizing AirPods/cable-related cords, consider the 40 Pcs Silicone Cable Ties (ID: B0B55TZLBR, rating 4.7). They’re reusable adjustable straps that help keep charging cables neatly fastened and organized.',
 'question': 'Can you recommend products for cleaning AirPods and organizing their cables?',
 'retrieved_context_ids': ['B0B7495RL6',
  'B0CF1WM24K',
  'B0BYYLJRHT',
  'B0B55TZLBR',
  'B0C6K1GQCF'],
 'retrieved_context': ['Cleaner Kit for AirPods Pro, 4 in 1 Earbuds Cleaning Pen, Bluetooth Headphone Cleaning Pen for Airpods, Airpods Pro 1 2 3 and Other Earphones, Keyboard, Mouse, Cellphones, Laptop, Camera (White) 【4-IN-1 DESIGN】The airpod cleaning kit is divided in

In [20]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [21]:
await ragas_faithfulness(result, "")

0.8333333333333334

In [22]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    
    return await scorer.single_turn_ascore(sample)

In [23]:
await ragas_relevancy(result, "")

np.float64(0.9187900085826651)

In [24]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [25]:
await ragas_context_precision_id_based(result, reference_output)

0.4

In [26]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [27]:
await ragas_context_recall_id_based(result, reference_output)

1.0